# uGMRT Pre-processing Notebook

Interactive wrapper around the `ugmrt_query` module for **manual, step-by-step** bandpass calibration and flagging.  
For large-scale iterative flagging runs use `v-based-outlier-detection.sh`.

---

### What this notebook does

| # | Step | Key function(s) |
|---|------|-----------------|
| 1 | Inspect the FITS file | `list_sources`, `list_antennas`, `list_frequency_properties`, `get_key_header_properties` |
| 2 | Source observability (El/Az, HA, PA, UV) | `query_source`, `print_source_query`, `plot_source_query` |
| 3 | Array layout | `plot_antenna_positions` |
| 4 | Session configuration | — |
| 5 | Row-index cache | `get_or_build_row_index` |
| 6 | Custom data selection | `DataSelection` |
| 7 | Bandpass solve (single iteration, non-destructive) | `derive_bandpass_iteration` |
| 8 | Per-antenna gain grid | `plot_bandpass_solution_grid` |
| 9 | Diagnostics & residual statistics | `run_bandpass_diagnostics` |
| 10 | Flag proposals | `propose_flag_updates_from_diagnostics` |
| 11 | Raw visibility inspection | `load_vis_for_source`, `plot_vis_amp_vs_uvdist/time/channel` |
| 12 | Corrected visibility inspection | `apply_bandpass_solution`, `plot_bandpass_corrected_vis_amp_vs_uvdist`, `plot_corrected_vector_avg_spectrum` |
| 13 | Flux model preview | `flux_model_3c48_perley_butler_2017`, `flux_model_3c286_perley_butler_2017` |

**All operations are non-destructive — input FITS is never modified on disk.**

## 1 · Setup

Reload `ugmrt_query` without restarting the kernel so edits to the module are picked up immediately.  
`%matplotlib inline` embeds figures; switch to `%matplotlib widget` for interactive pan/zoom (requires `ipympl`).

In [ ]:
%matplotlib inline

import importlib
import sys
import numpy as np
from pathlib import Path

if 'ugmrt_query' in sys.modules:
    importlib.reload(sys.modules['ugmrt_query'])
import ugmrt_query as q

print('ugmrt_query loaded')

## 2 · File Discovery

Quick inventory of the FITS file — no index build required.

| Function | What it returns |
|----------|-----------------|
| `get_key_header_properties(path)` | Telescope, date, N channels, N vis groups, N antennas, N sources |
| `list_sources(path)` | Source names, IDs, RA/Dec |
| `list_antennas(path)` | Antenna numbers, names, ECEF XYZ positions (m) |
| `list_frequency_properties(path)` | N channels, centre frequency, channel width, total bandwidth, per-channel frequencies |

**Tip:** run this cell first every session to confirm the file path is correct.

In [ ]:
CAL_FITS = Path('/Users/raj030/DATA/gmrt_40_014/data/40_014_25jul2021_gsb.FITS')

# ── Key header summary ────────────────────────────────────────────────────────
hdr = q.get_key_header_properties(CAL_FITS)
print('── Header ──────────────────────────────────────')
for k, v in hdr.items():
    print(f'  {k:<30s}: {v}')

# ── Sources ───────────────────────────────────────────────────────────────────
print()
print('── Sources ─────────────────────────────────────')
for s in q.list_sources(CAL_FITS):
    print(f"  id={s['source_id']:>2d}  {s['source_name']:<12s}  "
          f"RA={s.get('ra_deg', float('nan')):.4f}°  Dec={s.get('dec_deg', float('nan')):.4f}°")

# ── Antennas ──────────────────────────────────────────────────────────────────
print()
print('── Antennas ────────────────────────────────────')
ants = q.list_antennas(CAL_FITS)
for a in ants:
    print(f"  no={a['antenna_no']:>2d}  {a['name']:<6s}  "
          f"x={a['x_m']:+.1f}  y={a['y_m']:+.1f}  z={a['z_m']:+.1f}")

# ── Frequency properties ──────────────────────────────────────────────────────
print()
print('── Frequency ───────────────────────────────────')
fp = q.list_frequency_properties(CAL_FITS)
print(f"  nchan={fp['nchan']}  centre={fp['ref_freq_hz']/1e6:.3f} MHz")
print(f"  chan_width={fp['chan_width_hz']/1e3:.3f} kHz  BW≈{fp['bandwidth_hz_estimated']/1e6:.2f} MHz")
print(f"  freq range: {fp['freq_min_hz']/1e6:.3f} – {fp['freq_max_hz']/1e6:.3f} MHz")
print(f"  first 8 chan freqs (MHz): {[f'{f/1e6:.3f}' for f in fp['first_8_channel_freq_hz']]}")

## 3 · Source Observability

`query_source` extracts every observable-quality property from the FITS file:
elevation/azimuth track, hour angle, parallactic angle, UV coverage  
statistics, sensitivity estimate, and data-selection advice.

```python
result = q.query_source(fits_path_or_index, source, azel_time_step_s=60.0)
```

- Pass the FITS path or a pre-built row index (avoids re-opening the file).
- `print_source_query(result)` prints a human-readable summary.
- `plot_source_query(result)` produces a four-panel diagnostic figure:  
  top-left El/Az, top-right UV plane, bottom-left HA/PA, bottom-right baseline histogram.

**Key outputs used downstream:**

| Key | Use |
|-----|-----|
| `result['data_selection_advice']['recommended_elevation_min_deg']` | Set `SOLVE_ELEVATION_MIN_DEG` |
| `result['uv_coverage']['b_max_klambda']` | Upper limit for `uvrange_klambda` in `DataSelection` |
| `result['timing']['scans']` | Identify scan boundaries for time-range cuts |

In [ ]:
# ── Build row index once (reused for source query and all downstream steps) ──
# Adjust SOURCE_QUERY to the calibrator you want to inspect.
SOURCE_QUERY = '3C48'   # change to '3C286', 'SRCNAME', etc.

# Use CAL_FITS from the cell above, or re-define it here.
result = q.query_source(CAL_FITS, SOURCE_QUERY, azel_time_step_s=60.0)

# ── Printed summary ───────────────────────────────────────────────────────────
q.print_source_query(result)

# ── Diagnostic plot (El/Az, UV, HA/PA, baseline histogram) ───────────────────
fig = q.plot_source_query(result, figsize=(18, 14))
fig.tight_layout()

## 4 · Array Layout

`plot_antenna_positions` draws the GMRT array from the ENU coordinates in the AIPS AN table.

```python
fig = q.plot_antenna_positions(
    fits_path_or_index,           # FITS path or row index
    flagged_antenna_names=['C11'],# highlight flagged antennas in red
    highlight_pairs_closer_than_m=100.0,  # draw lines + standing-wave annotation
)
```

Useful to visually confirm which antennas are short-spaced (susceptible to standing waves)  
and which have already been flagged in a flag table.

In [ ]:
# Optionally mark currently flagged antennas from the session flag table.
import json as _json

_flagged_names = []
_flag_table_path = Path('/Users/raj030/DATA/gmrt_40_014/work/3c48_flag_table_session.json')
if _flag_table_path.exists():
    _ft = _json.loads(_flag_table_path.read_text())
    _flagged_names = list(_ft.get('bad_antennas', {}).keys())
    print('Flagged antennas from session table:', _flagged_names)

fig = q.plot_antenna_positions(
    CAL_FITS,
    flagged_antenna_names=_flagged_names,
    highlight_pairs_closer_than_m=200.0,
    title=f'GMRT array — {CAL_FITS.name}',
)

## 5 · Session Configuration

All tunable parameters live here.  Re-run this cell whenever you want to change source,  
iteration tag, flag policy, or thresholds — **without** re-running the row-index or solve cells.

**Paths are derived automatically from `SOURCE`** via `_src = SOURCE.lower()`, so switching  
source (e.g. `'3C286'`) automatically redirects all output files.

> **DRY_RUN_BANDPASS = True** means the solution is computed but not written to disk.  
> **DRY_RUN_FLAG_WRITE = True** means flag proposals are held in memory (`PENDING_FLAG_TABLES`), not written.  
> Set both to `False` only when you are happy with a solution and want to commit it.

In [ ]:
# ── Iteration book-keeping ────────────────────────────────────────────────────
ITER_TAG = 'iter00'

# ── File paths ────────────────────────────────────────────────────────────────
BASE_DIR = Path('/Users/raj030/DATA/gmrt_40_014')
WORK_DIR = BASE_DIR / 'work'
DATA_DIR = BASE_DIR / 'data'

CAL_FITS    = DATA_DIR / '40_014_25jul2021_gsb.FITS'
INDEX_CACHE = WORK_DIR / '40_014_25jul2021_gsb.row_index_cache.npz'

# Row-index cache validation mode:
#   'fast'      -> compare file size + modification time (default for most sessions)
#   'fast+sha'  -> fast check first; on mismatch, verify SHA before rebuild
#   'sha256'    -> always recompute SHA256 (slowest, strictest)
#   'none'      -> trust cache unconditionally
INDEX_VALIDATION_MODE = 'fast+sha'

# ── Source ────────────────────────────────────────────────────────────────────
# Registered calibrators: 3C48, 3C286 (both use Perley-Butler 2017 Table 2).
SOURCE = '3C48'

# Source-derived output file stems (change SOURCE above, paths follow automatically)
_src = SOURCE.lower()
FLAG_TABLE_BASE    = WORK_DIR / f'{_src}_flag_table.json'
FLAG_TABLE_SESSION = WORK_DIR / f'{_src}_flag_table_session.json'
FLAG_TABLE_PATHS   = [p for p in [FLAG_TABLE_BASE, FLAG_TABLE_SESSION] if p.exists()]

# In-memory (dry-run) flag proposals from previous iterations
if 'PENDING_FLAG_TABLES' not in globals():
    PENDING_FLAG_TABLES = []

USE_PENDING_FLAG_TABLES    = True
CLEAR_PENDING_FLAG_TABLES  = False
if CLEAR_PENDING_FLAG_TABLES:
    PENDING_FLAG_TABLES = []

DRY_RUN_BANDPASS  = True
DRY_RUN_FLAG_WRITE = True

# ── Solve options ─────────────────────────────────────────────────────────────
STOKES        = ('RR', 'LL')
CHAN_RANGE     = (64, 191)     # 0-based inclusive channel indices to solve over
MAX_ROWS_SOLVE = 150_000
SMOOTH_WINDOW  = 5             # window size for Re/Im bandpass smoothing
MIN_BASELINES  = 20

# Symmetrize per-sample flags across correlations at load time:
# whenever any correlation is natively flagged (weight ≤ 0) for a given
# (row, channel), all correlations are forced flagged at that point.
FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED = True

# ── Diagnostics options ───────────────────────────────────────────────────────
EXCLUDE_FOR_PLOTS           = []
SKIP_EDGE_CHANNELS          = (0, 0)
TOP_N                       = 12
DIAG_APPLY_FLAGS_ON_THE_FLY     = True
DIAG_SAVE_UNFLAGGED_COMPARISON  = False

# ── Outlier detection metrics ─────────────────────────────────────────────────
#
# OUTLIER_METRIC — signal(s) used to score each antenna/baseline.
#   'RR'  → |Re(V_RR^corrected) − S_ν|  Perley-Butler residual (registered sources only)
#   'LL'  → |Re(V_LL^corrected) − S_ν|  same
#   'V'   → |RR − LL|  Stokes-V proxy, sky-model-independent (works for any source)
#
# OUTLIER_METRIC_MERGE_STRATEGY — when multiple metrics are active:
#   'union'        → flag if threshold exceeded in ANY metric  (recommended)
#   'intersection' → flag only if exceeded in ALL metrics      (conservative)
OUTLIER_METRIC                = ('RR', 'LL', 'V')
OUTLIER_METRIC_MERGE_STRATEGY = 'union'

# ── Flag thresholds ───────────────────────────────────────────────────────────
# V (|RR−LL|) scores are much smaller than RR/LL residuals for clean data.
# A realistic bad antenna on an unpolarised calibrator produces ~5–50 Jy of
# differential signal, so V needs a much lower threshold.
ANTENNA_FLAG_THRESHOLD_JY  = {'RR': 180.0, 'LL': 180.0, 'V': 30.0}   # Jy
BASELINE_FLAG_THRESHOLD_JY = {'RR': 800.0, 'LL': 800.0, 'V': 150.0}  # Jy

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'SOURCE           : {SOURCE}')
print(f'ITER_TAG         : {ITER_TAG}')
print(f'CAL_FITS         : {CAL_FITS}')
print(f'INDEX_CACHE      : {INDEX_CACHE}')
print(f'FLAG_TABLE_BASE  : {FLAG_TABLE_BASE}')
print(f'FLAG_TABLE_SESSION: {FLAG_TABLE_SESSION}')
print(f'FLAG_TABLE_PATHS : {FLAG_TABLE_PATHS}')
print(f'Pending in-memory flag tables: {len(PENDING_FLAG_TABLES)}')
print(f'USE_PENDING_FLAG_TABLES : {USE_PENDING_FLAG_TABLES}')
print(f'DRY_RUN_BANDPASS : {DRY_RUN_BANDPASS}')
print(f'DRY_RUN_FLAG_WRITE: {DRY_RUN_FLAG_WRITE}')
print(f'FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED: {FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED}')
print(f'OUTLIER_METRIC   : {OUTLIER_METRIC}')
print(f'ANTENNA_FLAG_THRESHOLD_JY : {ANTENNA_FLAG_THRESHOLD_JY}')
print(f'BASELINE_FLAG_THRESHOLD_JY: {BASELINE_FLAG_THRESHOLD_JY}')

## 6 · Row-Index Cache

The row index is a compact binary cache (`~5–50 MB`) that maps every FITS vis-group row  
to its source ID, baseline pair, timestamp, and elevation.  It is built once and reused  
across all subsequent cells.

```python
row_index = q.get_or_build_row_index(
    CAL_FITS,
    cache_path=INDEX_CACHE,
    force_rebuild=False,           # set True to rebuild even if cache is fresh
    validation_mode='fast+sha',    # 'fast', 'fast+sha', 'sha256', or 'none'
    write_cache=True,
)
```

The returned dict contains `source_ranges`, `id_to_name`, `elevation_deg`, `jd`, etc.  
Everything that needs per-row metadata reads from this dict — the raw FITS file is  
only touched at initial build time.

In [ ]:
row_index = q.get_or_build_row_index(
    CAL_FITS,
    cache_path=INDEX_CACHE,
    force_rebuild=False,
    validation_mode=INDEX_VALIDATION_MODE,
    write_cache=True,
)

print('Index cache path  :', row_index.get('index_cache_path', INDEX_CACHE))
print('Source identity   :', row_index.get('source_identity'))
print('Source SHA256     :', row_index.get('source_sha256'))
print('Sources in index  :', {v: k for k, v in row_index['id_to_name'].items()})

## 7 · Data Selection

`DataSelection` is a reusable filter specification accepted by every loading and calibration function.

```python
sel = q.DataSelection(
    timerange    = ('2021-07-25 19:00:00', '2021-07-25 21:00:00'),  # ISO or JD float pair
    chan_range    = (64, 191),         # 0-based inclusive channel indices
    uvrange_m    = (None, None),       # UV distance range in metres
    uvrange_klambda = (2.0, 100.0),   # UV distance range in kλ (uses band centre)
    elevation_min_deg = 25.0,          # drop rows below this elevation
    elevation_max_deg = None,
    ant_list     = None,               # restrict to specific antenna numbers
)
```

All fields are optional — any combination can be used simultaneously.  
By default (no `DataSelection`) the standard `chan_range` / `max_rows` parameters apply.

In [ ]:
# ── Example: elevation + UV-distance cut ──────────────────────────────────────
sel = q.DataSelection(
    chan_range        = CHAN_RANGE,
    elevation_min_deg = 25.0,          # matches SOLVE_ELEVATION_MIN_DEG in the shell script
    uvrange_klambda   = (0.1, 150.0),  # exclude very short baselines
)
print('DataSelection:', sel)

# ── Example: narrow time window (first scan only) ──────────────────────────────
# sel_scan1 = q.DataSelection(
#     timerange = ('2021-07-25 19:00:00', '2021-07-25 20:00:00'),
#     chan_range = CHAN_RANGE,
# )
# print('Scan-1 selection:', sel_scan1)

# Pass sel to solve / load cells via the selection= parameter.
# Leave sel = None below to use the default chan_range / max_rows only.

## 8 · Bandpass Solve (single iteration)

`derive_bandpass_iteration` wraps the StefCal per-channel solver:

1. Loads visibilities from `CAL_FITS` using the row index.
2. Applies all on-disk flag tables **and** in-memory `PENDING_FLAG_TABLES` (if `USE_PENDING_FLAG_TABLES=True`).
3. Runs StefCal per channel, solves for complex antenna gains $G_{a,
u}$.
4. Smooths the solution: **Re** and **Im** parts are Gaussian-smoothed independently  
   (avoids branch-cut errors that plague unwrapped-phase smoothing).
5. Writes the `.npz` solution only when `DRY_RUN_BANDPASS=False`.

The returned `bandpass_run` dict contains:
- `bandpass_run['solution']` — the full solution dict (gains, freqs, antenna ids, …)
- `bandpass_run['bandpass_out']` — path where the solution was (or would be) written
- `bandpass_run['dry_run']` — whether the file write was skipped

**Registered calibrators for flux scaling:** 3C48, 3C286 (Perley-Butler 2017).

In [ ]:
_src = SOURCE.lower()
BANDPASS_OUT = WORK_DIR / f'{_src}_bandpass_25jul_gsb.npz'

active_pending = PENDING_FLAG_TABLES if USE_PENDING_FLAG_TABLES else []

bandpass_run = q.derive_bandpass_iteration(
    fits_path          = CAL_FITS,
    index              = row_index,
    bandpass_out       = BANDPASS_OUT,
    source             = SOURCE,
    stokes             = STOKES,
    chan_range         = CHAN_RANGE,
    max_rows           = MAX_ROWS_SOLVE,
    smooth_window      = SMOOTH_WINDOW,
    min_baselines      = MIN_BASELINES,
    ignore_autos       = True,
    flag_table_path    = FLAG_TABLE_PATHS if FLAG_TABLE_PATHS else None,
    flag_table         = active_pending if active_pending else None,
    flag_all_corrs_if_any_rawvis_flagged = FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED,
    iteration_tag      = ITER_TAG,
    dry_run            = DRY_RUN_BANDPASS,
)

bandpass_sol      = bandpass_run['solution']
bandpass_out_path = bandpass_run['bandpass_out']

print('dry_run            :', bandpass_run['dry_run'])
print('bandpass_out       :', bandpass_out_path)
print('on-disk flags used :', FLAG_TABLE_PATHS)
print('pending flags used :', len(active_pending))
print('merged flag count  :', bandpass_sol.get('flag_table_count', 0))
print('rows by flag tables:', bandpass_sol.get('solve_dropped_rows_by_flag_table', 0))

## 9 · Per-antenna Gain Grid

`plot_bandpass_solution_grid` renders every antenna on a rows × cols page.  
Each antenna cell shows:
- **Top panel** — amplitude $|G_{a,
u}|$ (raw + smoothed)
- **Bottom panel** — phase $\angle G_{a,
u}$ in degrees (raw + smoothed)

The reference antenna is pinned to phase = 0° across all channels (post-solve gauge choice).

```python
fig = q.plot_bandpass_solution_grid(
    solution,
    rows=6, cols=5,
    skip_edge_channels=(5, 5),     # ignore edge artefacts in auto-scaling
    phase_ylim=(-200, 200),
    canonical_antenna_names=sorted_names,  # fix panel order across iterations
)
```

In [ ]:
fig = q.plot_bandpass_solution_grid(
    bandpass_sol,
    rows  = 6,
    cols  = 5,
    figsize = (32, 40),
    skip_edge_channels = (5, 5),
    phase_ylim = (-200.0, 200.0),
    title = f'{SOURCE} bandpass solution | {ITER_TAG}',
)

## 10 · Diagnostics

`run_bandpass_diagnostics` applies the bandpass solution to the loaded visibilities  
and produces the multi-page diagnostic PDF / PNG:

- **Page 1** — corrected amp vs UV distance for each antenna (top N worst highlighted)
- **Page 2** — baselines ranked by outlier score
- **Page 3** — per-channel residuals vs model flux

Key outputs in the returned `diag` dict:

| Key | Description |
|-----|-------------|
| `diag['antenna_scores']` | Dict `{antenna_name: score_jy}` for the active metrics |
| `diag['baseline_scores']` | Dict `{(ant1, ant2): score_jy}` |
| `diag['freqs_hz']` / `diag['chan_mask']` | Frequency axis and valid-channel mask |
| `diag['plot_freq_min_mhz']` / `diag['plot_freq_max_mhz']` | Plotted frequency range |

In [ ]:
_src = SOURCE.lower()
DIAG_PLOT_BASE = WORK_DIR / f'{_src}_bandpass_diagnostics.png'

diag = q.run_bandpass_diagnostics(
    row_index,
    bandpass_sol,
    source    = SOURCE,
    chan_range = CHAN_RANGE,
    stokes    = STOKES,
    max_rows  = MAX_ROWS_SOLVE,
    exclude_antennas        = EXCLUDE_FOR_PLOTS,
    apply_flag_tables       = DIAG_APPLY_FLAGS_ON_THE_FLY,
    flag_table_path         = FLAG_TABLE_PATHS if FLAG_TABLE_PATHS else None,
    flag_table              = active_pending if active_pending else None,
    flag_all_corrs_if_any_rawvis_flagged = FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED,
    skip_edge_channels      = SKIP_EDGE_CHANNELS,
    top_n                   = TOP_N,
    ranking_metric          = OUTLIER_METRIC,
    title                   = f'{SOURCE} diagnostics | {ITER_TAG}',
    save_path               = DIAG_PLOT_BASE,
)

# ── Channel/frequency sanity check ────────────────────────────────────────────
_freqs = diag['freqs_hz']
_mask  = diag['chan_mask']
_chan_range_expected = CHAN_RANGE[1] - CHAN_RANGE[0] + 1
print(f'CHAN_RANGE          : {CHAN_RANGE}  ({_chan_range_expected} expected)')
print(f'Channels loaded     : {_freqs.size}')
print(f'Channels plotted    : {int(_mask.sum())}  (SKIP_EDGE_CHANNELS={SKIP_EDGE_CHANNELS})')
print(f'Freq range plotted  : {diag["plot_freq_min_mhz"]:.3f} – {diag["plot_freq_max_mhz"]:.3f} MHz')
print(f'Bandwidth plotted   : {diag["plot_freq_max_mhz"] - diag["plot_freq_min_mhz"]:.3f} MHz')

## 11 · Flag Proposals

`propose_flag_updates_from_diagnostics` converts outlier scores into flag-table entries.

```python
proposal = q.propose_flag_updates_from_diagnostics(
    diag,
    outlier_metric                = OUTLIER_METRIC,
    outlier_metric_merge_strategy = OUTLIER_METRIC_MERGE_STRATEGY,
    mode = 'both',           # 'antennas', 'baselines', or 'both'
    antenna_flag_threshold_jy  = ANTENNA_FLAG_THRESHOLD_JY,
    baseline_flag_threshold_jy = BASELINE_FLAG_THRESHOLD_JY,
    max_antennas_to_flag  = 1,   # protect against flagging too much in one shot
    max_baselines_to_flag = 6,
)
```

When `DRY_RUN_FLAG_WRITE=True`, the proposed entries are appended to `PENDING_FLAG_TABLES`  
so they are available to the next solve iteration **without touching disk**.  
Set `DRY_RUN_FLAG_WRITE=False` to write straight to `FLAG_TABLE_SESSION`.

In [ ]:
proposal = q.propose_flag_updates_from_diagnostics(
    diag,
    outlier_metric                = OUTLIER_METRIC,
    outlier_metric_merge_strategy = OUTLIER_METRIC_MERGE_STRATEGY,
    mode = 'both',
    antenna_flag_threshold_jy  = ANTENNA_FLAG_THRESHOLD_JY,
    baseline_flag_threshold_jy = BASELINE_FLAG_THRESHOLD_JY,
    max_antennas_to_flag       = 1,
    max_baselines_to_flag      = 6,
)

# proposal keys: 'proposal' (flag table dict), 'candidate_antennas', 'candidate_baselines'
_src = SOURCE.lower()
FLAG_TABLE_SESSION = WORK_DIR / f'{_src}_flag_table_session.json'

_has_updates = bool(proposal.get('candidate_antennas')) or bool(proposal.get('candidate_baselines'))
if not DRY_RUN_FLAG_WRITE and _has_updates:
    q.save_flag_table(proposal['proposal'], FLAG_TABLE_SESSION)
    print('Flag table written to:', FLAG_TABLE_SESSION)
elif _has_updates:
    PENDING_FLAG_TABLES.append(proposal['proposal'])
    print(f'Proposal held in memory (pending count: {len(PENDING_FLAG_TABLES)})')
else:
    print('No flag updates proposed.')

print('Candidate antennas :', proposal.get('candidate_antennas', []))
print('Candidate baselines:', proposal.get('candidate_baselines', []))

## 12 · Raw Visibility Inspection

`load_vis_for_source` loads a block of raw visibilities into memory as a structured dict.  
Pass a `DataSelection` to apply time/UV/elevation cuts before loading.

Three complementary views:

| Function | X axis | Notes |
|----------|--------|-------|
| `plot_vis_amp_vs_uvdist(vis)` | UV distance (kλ) | Shows amplitude envelope → source structure |
| `plot_vis_amp_vs_time(vis)` | Time from start (min) | Shows time-variable RFI or elevation effects |
| `plot_vis_amp_vs_channel(vis)` | Frequency (MHz) | Shows channel-specific RFI or roll-off |

Set `show_phase=True` on any of the above to add a phase panel below the amplitude panel.  
Use `amp_ylim=(0, X)` to cap the amplitude axis for zooming into the noise floor.

In [ ]:
# Load raw visibilities (pre-bandpass, pre-correction)
vis_raw = q.load_vis_for_source(
    row_index,
    source    = SOURCE,
    chan_range = CHAN_RANGE,
    stokes    = STOKES,
    max_rows  = 60_000,
    flag_all_corrs_if_any_rawvis_flagged = FLAG_ALL_CORRS_IF_ANY_RAWVIS_FLAGGED,
)

print(f'Loaded: {vis_raw["amp"].shape[0]} rows × {vis_raw["amp"].shape[1]} channels × {vis_raw["amp"].shape[2]} pols')
print(f'Freq range: {vis_raw["freqs_hz"].min()/1e6:.3f} – {vis_raw["freqs_hz"].max()/1e6:.3f} MHz')

# ── Amplitude vs UV distance ──────────────────────────────────────────────────
q.plot_vis_amp_vs_uvdist(
    vis_raw,
    title     = f'{SOURCE} raw vis — amp vs UV distance | {ITER_TAG}',
    show_phase = False,
    alpha     = 0.10,
)

# ── Amplitude vs time ─────────────────────────────────────────────────────────
q.plot_vis_amp_vs_time(
    vis_raw,
    title     = f'{SOURCE} raw vis — amp vs time | {ITER_TAG}',
    show_phase = False,
    alpha     = 0.10,
)

# ── Amplitude vs channel ──────────────────────────────────────────────────────
q.plot_vis_amp_vs_channel(
    vis_raw,
    title     = f'{SOURCE} raw vis — amp vs channel | {ITER_TAG}',
    show_phase = False,
    alpha     = 0.10,
)

## 13 · Corrected Visibility Inspection

After a bandpass solve, apply the solution in memory and inspect the corrected visibilities.

```python
corrected = q.apply_bandpass_solution(vis_raw, bandpass_sol)
# corrected has: 'vis_complex_corrected', 'amp_corrected', 'phase_deg_corrected', …
```

Two convenience plot wrappers:

| Function | What it shows |
|----------|---------------|
| `plot_bandpass_corrected_vis_amp_vs_uvdist(vis, sol)` | Corrected amplitudes vs UV distance → should be flat for an unresolved calibrator |
| `plot_corrected_vector_avg_spectrum(vis, sol)` | Vector-averaged Re(V) vs Perley-Butler 2017 flux model + residuals → residuals should be near zero for a clean calibrator |

A flat UV-distance plot and near-zero residuals mean the bandpass is correct  
to within noise. Structured residuals point to a bad antenna or RFI.

In [ ]:
# Corrected amplitude vs UV distance
q.plot_bandpass_corrected_vis_amp_vs_uvdist(
    vis_raw,
    bandpass_sol,
    title          = f'{SOURCE} corrected vis — amp vs UV dist | {ITER_TAG}',
    exclude_antennas = EXCLUDE_FOR_PLOTS,
    show_phase     = False,
    alpha          = 0.10,
)

# Vector-averaged spectrum vs Perley-Butler 2017 model
_src = SOURCE.lower()
GAIN_PLOT_BASE = WORK_DIR / f'{_src}_corrected_spectrum_{ITER_TAG}.png'

q.plot_corrected_vector_avg_spectrum(
    vis_raw,
    bandpass_sol,
    title            = f'{SOURCE} corrected vector-avg spectrum | {ITER_TAG}',
    exclude_antennas = EXCLUDE_FOR_PLOTS,
    skip_edge_channels = SKIP_EDGE_CHANNELS,
    save_path        = GAIN_PLOT_BASE if not DRY_RUN_BANDPASS else None,
)

## 14 · Flux Model Reference

Spot-check the Perley-Butler 2017 flux models for both registered calibrators.  
These are polynomial models in $\log_{10}(\nu / \text{GHz})$:

$$
\log_{10} S = a_0 + a_1 \log_{10}\nu + a_2 (\log_{10}\nu)^2 + a_3 (\log_{10}\nu)^3
$$

Valid frequency range: **50 MHz – 50 GHz** (Table 2, Perley & Butler 2017).

| Calibrator | $a_0$ | $a_1$ | $a_2$ | $a_3$ |
|------------|-------|-------|-------|-------|
| 3C48  | 1.3253 | −0.7553 | −0.1914 |  0.0498 |
| 3C286 | 1.2481 | −0.4507 | −0.1798 |  0.0357 |

At 323 MHz: **3C48 ≈ 44 Jy**, **3C286 ≈ 26 Jy**.

In [ ]:
import matplotlib.pyplot as plt

freqs_hz  = np.linspace(50e6, 900e6, 500)
freqs_mhz = freqs_hz / 1e6

flux_3c48  = q.flux_model_3c48_perley_butler_2017(freqs_hz)
flux_3c286 = q.flux_model_3c286_perley_butler_2017(freqs_hz)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(freqs_mhz, flux_3c48,  lw=2.0, label='3C48  (PB2017)')
ax.plot(freqs_mhz, flux_3c286, lw=2.0, label='3C286 (PB2017)')
ax.axvline(freqs_hz[0] / 1e6, color='grey', lw=0.8, ls='--')
ax.set_xlabel('Frequency (MHz)')
ax.set_ylabel('Flux density (Jy)')
ax.set_title('Perley-Butler 2017 flux models — 50 MHz to 900 MHz')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()

# Print values at the GMRT band centre (≈ 323 MHz)
_f0 = 323e6
print(f'At {_f0/1e6:.0f} MHz:')
print(f'  3C48  = {q.flux_model_3c48_perley_butler_2017(np.array([_f0]))[0]:.2f} Jy')
print(f'  3C286 = {q.flux_model_3c286_perley_butler_2017(np.array([_f0]))[0]:.2f} Jy')

## Iteration Loop

To run the next manual iteration:

1. Set `ITER_TAG` to the next value (e.g. `iter01`, `iter02`).
2. Re-run the **Configuration** cell (§5) — this updates `_src`-derived paths.
3. Re-run the **Bandpass Solve** cell (§8).
4. Re-run the **Diagnostics** cell (§10) and **Flag Proposals** cell (§11).
5. If the proposal looks good, set `DRY_RUN_FLAG_WRITE=False` and re-run §11  
   to commit the flag entries to `FLAG_TABLE_SESSION`.
6. Repeat from step 1 with the new flag table active.

Flag table priority (applied in this order during each solve):
1. On-disk tables in `FLAG_TABLE_PATHS`
2. In-memory `PENDING_FLAG_TABLES` (only when `USE_PENDING_FLAG_TABLES=True`)

To discard accumulated in-memory proposals: set `CLEAR_PENDING_FLAG_TABLES=True`  
and re-run the Configuration cell.

---

For fully automated multi-iteration flagging, use the shell script:
```bash
./v-based-outlier-detection.sh \
  --auto --set "SOURCE='3C48'" \
  --n-iters 30 \
  --set "CONVERGENCE_EPSILON=0.005" \
  --set "FLAG_WHAT_TO_FLAG='baselines'" \
  --set "SOLVE_ELEVATION_MIN_DEG=25.0"
```